# iTantra Tamil Gate B: baseline and bounded training pilot

**Prepared 4 October 2026. Actual Kaggle runtime, baseline and training: Not run. Phone accuracy: Not verified.**

This notebook includes the saved-data/GPU preflight. You do not need to run the separate preflight notebook first or repeat Gate A.

1. Create a **new private** Kaggle notebook and import this `.ipynb` using **File > Import Notebook**.
2. **Add Input > Notebook > Your Work**: attach `aviraltindori/notebook2efe523570`, successful **Version #1**, containing `itantra`, `itantra-kathbath`, `itantra-noisy` and `itantra_gate_a_report.json`. The `language 2` input alone is insufficient. Attach exactly one complete Gate A output.
3. Settings: Internet **on**, CUDA GPU **on** (one GPU with at least 14 GiB reported memory). Enable the existing **HF_TOKEN** secret for this notebook. Never paste its value into code. Accept the [Tamil checkpoint access terms](https://huggingface.co/ai4bharat/indicconformer_stt_ta_hybrid_ctc_rnnt_large) on your Hugging Face account.
4. Run all cells once. On failure, **stop** and download `itantra_tamil_pilot_reports.zip` from Output. Do not restart or run Gate A again. A failure report is useful.
5. On success, download that same report ZIP and share it. Preserve the notebook outputs using a save option that includes current outputs. Do not choose a full rerun just to download reports.

The experiment measures the original CTC/RNNT baseline, then runs **20 optimizer steps per run**, freeze-lower vs full encoder, with actual seeds **17 and 29**. It uses only 128 training and 64 development clips from the reviewed splits, with unchanged speaker separation. It does **not** run the full 10-hour training trial, select on held-out tests, export, or update the APK. This pilot cannot establish improved general accuracy or <10% WER.

Runtime packages and source checkpoint download over Internet during preparation. The installed app's offline inference/model-download design is unchanged.


In [ ]:
import os, sys, json, subprocess, shutil, zipfile, time
from pathlib import Path
assert sys.version_info >= (3, 11), 'Use a Kaggle Python 3.11+ kernel'
work = Path('/kaggle/working/itantra-tamil-pilot')
work.mkdir(parents=True, exist_ok=True)
logs = work / 'logs'
logs.mkdir(exist_ok=True)
summary = {'status': 'Prepared; runtime/baseline/pilot not run', 'language': 'ta',
           'full_trial': 'Not run', 'heldout_after_training': 'Not run', 'export': 'Not run',
           'phone_accuracy': 'Not verified', 'app_promotion': 'Not done'}

def pack_reports():
    (work/'handoff-report.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    archive = Path('/kaggle/working/itantra_tamil_pilot_reports.zip')
    files = [*work.glob('*report.json'), work/'pilot-plan.json', work/'constraints.txt',
             *work.glob('kathbath_*.py'), *logs.glob('*.log'),
             *work.glob('pilot-*/report.json'), *work.glob('pilot-*/*config.yaml'),
             *work.glob('pilot-*/metrics/version_*/*.csv')]
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in sorted(set(files)):
            if path.is_file():
                z.write(path, path.relative_to(work).as_posix())
    print('Download this report ZIP from Output:', archive)

def command(args, name):
    token = os.environ.get('HF_TOKEN') or '<no-token>'
    path = logs / (name + '.log')
    with path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(list(map(str, args)), stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   env=dict(os.environ, PYTHONUNBUFFERED='1', PYTHONFAULTHANDLER='1'))
        for line in process.stdout:
            clean = line.replace(token, '[REDACTED]')
            log.write(clean)
            log.flush()
            print(clean, end='', flush=True)
        returncode = process.wait()
        summary.setdefault('process_exit_codes', {})[name] = returncode
        footer = f'\n[process_exit] {name}: returncode={returncode}\n'
        log.write(footer)
        print(footer, end='', flush=True)
        if returncode:
            raise RuntimeError(f'{name} failed (returncode={returncode}); see {path.name} and the report ZIP')

def stage(name, action):
    summary['current_stage'] = name
    try:
        action()
        summary[name] = 'Completed'
    except Exception as error:
        summary[name] = 'Failed'
        summary['status'] = 'Stopped; review failure report'
        token = os.environ.get('HF_TOKEN') or '<no-token>'
        summary['error'] = {'type': type(error).__name__, 'message': str(error).replace(token, '[REDACTED]')[:2000]}
        raise RuntimeError(summary['error']['message']) from None
    finally:
        pack_reports()

(work / 'kathbath_gate_b_data.py').write_text('"""Recheck saved Gate A data without downloading or copying training WAVs.\n\nThe Kaggle preflight notebook embeds this module, so importing the notebook is\nthe only new upload needed. This module neither trains nor installs packages.\n"""\n\nimport hashlib\nimport json\nimport math\nimport shutil\nimport subprocess\nimport sys\nimport wave\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath, PureWindowsPath\n\nREVIEWED_REPORT_SHA256 = "73ea703471c99427ce0fbe9757c9854a2607bfe9bb35d6eeda4772a252384451"\n\n\ndef digest(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef portable_relative_path(relative):\n    if not isinstance(relative, str) or not relative or "\\x00" in relative:\n        raise RuntimeError(f"Unsafe saved path: {relative}")\n    # The original benchmark manifests were generated on Windows. Interpret\n    # both separators, without editing the checksum-protected source manifests.\n    part = PurePosixPath(relative.replace("\\\\", "/"))\n    windows = PureWindowsPath(relative)\n    if part.is_absolute() or windows.drive or windows.root or ".." in part.parts or ":" in relative:\n        raise RuntimeError(f"Unsafe saved path: {relative}")\n    return part\n\n\ndef contained_file(root, relative):\n    part = portable_relative_path(relative)\n    root = Path(root).resolve()\n    path = (root / part).resolve(strict=True)\n    if not path.is_relative_to(root) or not path.is_file():\n        raise RuntimeError(f"Saved file escapes its input root: {relative}")\n    return path\n\n\ndef saved_audio_path(path, saved_root):\n    original = PurePosixPath(path.replace("\\\\", "/"))\n    try:\n        relative = original.relative_to("/kaggle/working")\n    except ValueError as exc:\n        raise RuntimeError(f"Audio path is not from the reviewed Gate A session: {path}") from exc\n    if not relative.parts or relative.parts[0] not in ("itantra-kathbath", "itantra-noisy"):\n        raise RuntimeError(f"Unexpected Gate A audio folder: {path}")\n    return contained_file(saved_root, relative.as_posix())\n\n\ndef find_saved_root(input_root):\n    candidates = [p.parent for p in Path(input_root).rglob("itantra_gate_a_report.json")\n                  if (p.parent / "itantra-kathbath").is_dir()\n                  and (p.parent / "itantra-noisy").is_dir()\n                  and (p.parent / "itantra/tools/stt_training").is_dir()]\n    if len(candidates) != 1:\n        raise RuntimeError("Attach exactly one completed Gate A notebook output using "\n                           f"Add Input > Notebook > Your Work; found {len(candidates)} complete roots")\n    return candidates[0].resolve()\n\n\ndef wav_duration(path):\n    with wave.open(str(path), "rb") as audio:\n        if (audio.getframerate(), audio.getnchannels(), audio.getsampwidth(), audio.getcomptype()) != (\n                16000, 1, 2, "NONE"):\n            raise RuntimeError(f"Expected 16-kHz mono PCM16 WAV: {path}")\n        if not audio.getnframes():\n            raise RuntimeError(f"Empty WAV: {path}")\n        audio.setpos(audio.getnframes() - 1)\n        if len(audio.readframes(1)) != 2:\n            raise RuntimeError(f"Truncated WAV payload: {path}")\n        return audio.getnframes() / 16000\n\n\ndef prepare(saved_root, repo, language="ta", reviewed_sha=REVIEWED_REPORT_SHA256):\n    if language != "ta":\n        raise RuntimeError("This preflight is Tamil only; Telugu follows the reviewed Tamil trial")\n    saved_root, repo = Path(saved_root).resolve(), Path(repo).resolve()\n    if repo == saved_root or repo.is_relative_to(saved_root):\n        raise RuntimeError("Working output must be separate from saved input")\n    report_path = contained_file(saved_root, "itantra_gate_a_report.json")\n    if digest(report_path) != reviewed_sha:\n        raise RuntimeError("Saved Gate A report differs from the report reviewed on 4 October 2026")\n    report = json.loads(report_path.read_text(encoding="utf-8"))\n    entry = report["languages"][language]\n    audit, plan = entry["audit"], entry["plan"]\n    if report["status"] != "Gate A only; training not run":\n        raise RuntimeError("Unexpected Gate A report status")\n\n    source_repo = saved_root / "itantra"\n    provenance = json.loads(contained_file(source_repo, "bundle-provenance.json").read_text(encoding="utf-8"))\n    if provenance != report["bundle_provenance"]:\n        raise RuntimeError("Saved bundle provenance differs from the reviewed report")\n    # Verify the original evaluation audio and tools before executing copied code.\n    for name, expected in provenance["working_files"].items():\n        if digest(contained_file(source_repo, name)) != expected:\n            raise RuntimeError(f"Original bundle file changed: {name}")\n    tools = repo / "tools"\n    tools.mkdir(parents=True, exist_ok=True)\n    for name in provenance["working_files"]:\n        part = portable_relative_path(name)\n        if part.parts[0] == "tools" and part.suffix == ".py":\n            destination = tools / part.name\n            shutil.copyfile(contained_file(source_repo, name), destination)\n    sys.path.insert(0, str(tools))\n    from check_speech_leakage import check_rows, text_hash\n    from prepare_speech_finetune import native_text\n\n    data = tools / "stt_training"\n    data.mkdir(exist_ok=True)\n    index_path = contained_file(source_repo, f"tools/stt_training/{language}-combined-protected.json")\n    index = json.loads(index_path.read_text(encoding="utf-8"))\n    base = json.loads(contained_file(source_repo, f"tools/stt_training/{language}-protected.json").read_text(encoding="utf-8"))\n    if index.get("language") != language or set(index.get("splits", {})) != {"validation", "test"}:\n        raise RuntimeError("Wrong language or incomplete protected FLEURS splits")\n    extra = index.get("extra_protection", {})\n    if (extra.get("dataset_revision") != plan["revision"]\n            or extra.get("kathbath_noisy_rows") != entry["noisy_test_count"]\n            or extra.get("kathbath_nontrain_rows", {}).get("valid", 0) <= 0):\n        raise RuntimeError("Combined protection lacks the reviewed Kathbath valid/noisy scope")\n    for key in ("text_sha256", "source_ids", "audio_sha256"):\n        if not set(base[key]).issubset(index[key]):\n            raise RuntimeError(f"Combined protection dropped FLEURS entries: {key}")\n\n    sides, speaker_seconds, audio_hashes = {}, defaultdict(float), {}\n    for split in ("train", "dev"):\n        manifest = contained_file(source_repo, f"tools/stt_training/manifests/{language}-{split}.jsonl")\n        rows = [json.loads(line) for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()]\n        if len(rows) != audit["counts"][split]:\n            raise RuntimeError(f"{split} clip count differs from Gate A")\n        for row in rows:\n            row["audio_filepath"] = str(saved_audio_path(row["audio_filepath"], saved_root))\n            duration = wav_duration(row["audio_filepath"])\n            if not 1 <= duration <= 20 or abs(duration - row["duration"]) > 1 / 16000:\n                raise RuntimeError(f"Invalid measured duration: {row[\'source_id\']}")\n            if not native_text(language, row["text"]):\n                raise RuntimeError(f"Non-native reference transcript: {row[\'source_id\']}")\n            if (row["source"], row["source_version"], row["license"]) != (\n                    "ai4bharat/Kathbath", plan["revision"], "CC-BY-4.0"):\n                raise RuntimeError(f"Source provenance differs: {row[\'source_id\']}")\n            speaker_seconds[row["speaker_group"]] += duration\n        if not math.isclose(sum(r["duration"] for r in rows) / 3600, audit["hours"][split], abs_tol=1e-8):\n            raise RuntimeError(f"{split} hours differ from Gate A")\n        if len({r["speaker_group"] for r in rows}) != audit["speaker_groups_by_split"][split]:\n            raise RuntimeError(f"{split} speaker count differs from Gate A")\n        errors = check_rows(index, rows)\n        if errors:\n            raise RuntimeError(f"{split} leakage check failed: {\'; \'.join(errors[:5])}")\n        sides[split] = rows\n        audio_hashes[split] = {digest(r["audio_filepath"]) for r in rows}\n    train, dev = sides["train"], sides["dev"]\n    if ({r["speaker_group"] for r in train} & {r["speaker_group"] for r in dev}\n            or {r["source_id"] for r in train} & {r["source_id"] for r in dev}\n            or audio_hashes["train"] & audio_hashes["dev"]):\n        raise RuntimeError("Train/dev speaker, source ID or audio overlap")\n    if max(speaker_seconds.values()) > audit["speaker_cap_minutes"] * 60 + 1 / 16000:\n        raise RuntimeError("Measured speaker duration exceeds the Gate A cap")\n    all_rows = train + dev\n    if len({r["source_id"] for r in all_rows}) != len(all_rows):\n        raise RuntimeError("Duplicate training/dev source IDs")\n    if dict(Counter(r["gender"] for r in all_rows)) != audit["gender_clips"]:\n        raise RuntimeError("Gender counts differ from Gate A")\n\n    noisy_path = contained_file(saved_root, f"itantra-noisy/noisy-{language}-test-unknown.json")\n    noisy = json.loads(noisy_path.read_text(encoding="utf-8"))\n    if len(noisy) != entry["noisy_test_count"]:\n        raise RuntimeError("Noisy test count differs from Gate A")\n    for row in noisy:\n        if row["language"] != language or row["split"] != "test_unknown":\n            raise RuntimeError("Wrong noisy test language/split")\n        row["file"] = str(saved_audio_path(row["file"], saved_root))\n        wav_duration(row["file"])\n        if digest(row["file"]) != row["sha256"]:\n            raise RuntimeError(f"Noisy test WAV checksum mismatch: {row[\'source_id\']}")\n        if (row["source_id"] not in index["source_ids"] or row["sha256"] not in index["audio_sha256"]\n                or text_hash(row["reference"]) not in index["text_sha256"]):\n            raise RuntimeError("Noisy test row is absent from the combined protection index")\n\n    # Absolute file fields also work with the existing ROOT / row[\'file\'] evaluators.\n    # Rewrite only small manifests; leave all evaluation WAVs in read-only input.\n    for directory, count in (("five-language-validation", 30), ("five-language-test", 100)):\n        name = f"manifest-{language}-{count}.json"\n        original = contained_file(source_repo, f"tools/stt_models/{directory}/{name}")\n        records = json.loads(original.read_text(encoding="utf-8"))\n        if len(records) != count:\n            raise RuntimeError(f"Expected {count} protected FLEURS evaluation rows")\n        for row in records:\n            row["file"] = str(contained_file(source_repo, row["file"]))\n            if digest(row["file"]) != row["sha256"]:\n                raise RuntimeError("FLEURS evaluation WAV checksum mismatch")\n        destination = tools / "stt_models" / directory\n        destination.mkdir(parents=True, exist_ok=True)\n        (destination / name).write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")\n    manifest_root = data / "manifests"\n    manifest_root.mkdir(exist_ok=True)\n    for split, rows in sides.items():\n        (manifest_root / f"{language}-{split}.jsonl").write_text(\n            "".join(json.dumps(row, ensure_ascii=False) + "\\n" for row in rows), encoding="utf-8")\n    protected = data / f"{language}-combined-protected.json"\n    shutil.copyfile(index_path, protected)\n    rebased_noisy = data / f"noisy-{language}-test-unknown.json"\n    rebased_noisy.write_text(json.dumps(noisy, ensure_ascii=False, indent=2), encoding="utf-8")\n    check = subprocess.run([sys.executable, str(tools / "check_speech_leakage.py"), "check",\n                            "--language", language, "--protected", str(protected), "--manifest",\n                            str(manifest_root / f"{language}-train.jsonl"),\n                            str(manifest_root / f"{language}-dev.jsonl")],\n                           check=True, capture_output=True, text=True)\n    return {"status": "Saved Tamil data checks passed; training not run", "language": language,\n            "saved_data_verified": True,\n            "gate_a_report_sha256": digest(report_path), "saved_root": str(saved_root), "repo": str(repo),\n            "bundle_files_verified": len(provenance["working_files"]), "counts": audit["counts"],\n            "hours": audit["hours"], "speakers": audit["speaker_groups_by_split"],\n            "max_speaker_seconds": max(speaker_seconds.values()), "leakage_output": check.stdout.strip(),\n            "noisy_test_count": len(noisy), "training_audio_copied": False,\n            "post_training_wer": "Not run", "phone_accuracy": "Not verified"}\n', encoding='utf-8')

(work/'kathbath_tamil_pilot.py').write_text('"""Bounded Tamil Gate B baseline/pilot. NeMo imports occur only on the CUDA host.\n\nThis is a training-readiness experiment, not the 10-hour training trial or an\napp update. The notebook embeds this file and the existing saved-data guard.\n"""\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport statistics\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom kathbath_gate_b_data import digest\n\nNEMO_REVISION = "8dce88cf8e94963e2033c3137f7b9993b51db88a"\nMODEL_REPO = "ai4bharat/indicconformer_stt_ta_hybrid_ctc_rnnt_large"\nMODEL_REVISION = "8c31aa8d04964b8fc87e4eaaee7916f7d2c024da"\nMODEL_FILE = "indicconformer_stt_ta_hybrid_rnnt_large.nemo"\nREFERENCE_WER = 0.2182741116751269\nBASELINE_TOLERANCE = 0.03\nMAX_STEPS = 20\nSEEDS = (17, 29)\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n",\n                         encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_rows(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]\n\n\ndef subset(rows, count):\n    """Fixed speaker round-robin, never using transcripts/WER to select clips."""\n    groups = defaultdict(list)\n    for row in rows:\n        if 1 <= row["duration"] <= 10:\n            groups[row["speaker_group"]].append(row)\n    rng = random.Random(20261004)\n    names = sorted(groups)\n    rng.shuffle(names)\n    for name in names:\n        groups[name].sort(key=lambda row: row["source_id"])\n        rng.shuffle(groups[name])\n    selected = []\n    while len(selected) < count and any(groups.values()):\n        for name in names:\n            if groups[name] and len(selected) < count:\n                selected.append(groups[name].pop())\n    if len(selected) != count:\n        raise RuntimeError(f"Need {count} eligible TRAIN/DEV clips; found {len(selected)}")\n    return selected\n\n\ndef make_plan(repo, data_report):\n    repo = Path(repo)\n    if data_report["language"] != "ta" or not data_report["status"].startswith("Saved Tamil data checks passed"):\n        raise RuntimeError("A successful Tamil saved-data check is required")\n    data = repo / "tools/stt_training"\n    full = {s: read_rows(data / f"manifests/ta-{s}.jsonl") for s in ("train", "dev")}\n    chosen = {"train": subset(full["train"], 128), "dev": subset(full["dev"], 64)}\n    if ({r["speaker_group"] for r in chosen["train"]} & {r["speaker_group"] for r in chosen["dev"]}\n            or {r["source_id"] for r in chosen["train"]} & {r["source_id"] for r in chosen["dev"]}):\n        raise RuntimeError("Pilot TRAIN/DEV overlap")\n    manifests = {}\n    pilot = repo / "pilot"\n    pilot.mkdir(exist_ok=True)\n    for side, rows in chosen.items():\n        path = pilot / f"ta-{side}.jsonl"\n        # The fork\'s multilingual tokenizer reads `lang`, not `language_id`.\n        path.write_text("".join(json.dumps(dict(row, lang="ta"), ensure_ascii=False) + "\\n" for row in rows),\n                        encoding="utf-8")\n        manifests[side] = str(path.resolve())\n    protected = data / "ta-combined-protected.json"\n    command = [sys.executable, str(repo / "tools/check_speech_leakage.py"), "check", "--language", "ta",\n               "--protected", str(protected), "--manifest", *manifests.values()]\n    checked = subprocess.run(command, check=True, capture_output=True, text=True)\n    paths = [*map(Path, manifests.values()), *sorted((repo / "tools").glob("*.py")), protected,\n             data / "manifests/ta-train.jsonl", data / "manifests/ta-dev.jsonl",\n             data / "noisy-ta-test-unknown.json",\n             repo / "tools/stt_models/five-language-validation/manifest-ta-30.json",\n             repo / "tools/stt_models/five-language-test/manifest-ta-100.json"]\n    plan = {"status": "Pilot prepared; baseline and training not run", "language": "ta",\n            "gate_a_report_sha256": data_report["gate_a_report_sha256"], "manifests": manifests,\n            "sealed_files": {str(p.resolve()): digest(p) for p in paths},\n            "sealed_pilot_audio": {r["audio_filepath"]: digest(r["audio_filepath"])\n                                   for rows in chosen.values() for r in rows},\n            "counts": {s: len(rows) for s, rows in chosen.items()},\n            "hours": {s: sum(r["duration"] for r in rows) / 3600 for s, rows in chosen.items()},\n            "speakers": {s: len({r["speaker_group"] for r in rows}) for s, rows in chosen.items()},\n            "seeds": list(SEEDS), "modes": ["freeze-lower", "full"], "max_steps_per_run": MAX_STEPS,\n            "leakage_output": checked.stdout.strip(), "phone_accuracy": "Not verified",\n            "full_trial": "Not run", "heldout_after_training": "Not run", "app_promotion": "Not done"}\n    write_json(repo / "pilot-plan.json", plan)\n    return plan\n\n\ndef verify_plan(repo):\n    plan = json.loads((Path(repo) / "pilot-plan.json").read_text(encoding="utf-8"))\n    if plan["language"] != "ta" or plan["max_steps_per_run"] != MAX_STEPS or plan["seeds"] != list(SEEDS):\n        raise RuntimeError("Unexpected pilot scope; do not expand the reviewed trial")\n    for path, expected in plan["sealed_files"].items():\n        if digest(path) != expected:\n            raise RuntimeError(f"Prepared data changed: {path}")\n    for path, expected in plan["sealed_pilot_audio"].items():\n        if digest(path) != expected:\n            raise RuntimeError(f"Pilot audio changed: {path}")\n    # Detect audio changes after preflight as well as modified manifests.\n    sys.path.insert(0, str(Path(repo) / "tools"))\n    from check_speech_leakage import check_rows\n    index = json.loads((Path(repo) / "tools/stt_training/ta-combined-protected.json").read_text(encoding="utf-8"))\n    for path in plan["manifests"].values():\n        rows = read_rows(path)\n        if any(r.get("lang") != "ta" for r in rows):\n            raise RuntimeError("Pilot rows must explicitly pass Tamil to the tokenizer")\n        errors = check_rows(index, rows)\n        if errors:\n            raise RuntimeError("Pilot leakage: " + "; ".join(errors[:4]))\n    return plan\n\n\ndef completed_pilot(repo, mode, seed):\n    """Skip a completed run only when its measured evidence matches this trial."""\n    repo = Path(repo)\n    path = repo / f"pilot-{mode}-{seed}/report.json"\n    if not path.exists():\n        return None\n    result = json.loads(path.read_text(encoding="utf-8"))\n    valid = (result.get("status") == "Bounded pilot executed; full trial and held-out comparison not run"\n             and not result.get("error") and result.get("weights_changed") is True\n             and result.get("language") == "ta" and result.get("mode") == mode\n             and result.get("seed") == seed and seed in SEEDS and mode in ("freeze-lower", "full")\n             and result.get("max_steps") == MAX_STEPS and result.get("training_precision") == "32-true"\n             and isinstance(result.get("global_step"), int) and 0 < result["global_step"] <= MAX_STEPS\n             and result.get("plan_sha256") == digest(repo / "pilot-plan.json")\n             and result.get("baseline_report_sha256") == digest(repo / "baseline-report.json")\n             and (path.parent / "last-pilot.ckpt").is_file())\n    history = result.get("history", [])\n    valid = (valid and any("dev_ctc_wer" in row for row in history)\n             and any("loss" in row for row in history)\n             and all(math.isfinite(float(row[key])) and float(row[key]) >= 0\n                     for row in history for key in ("loss", "dev_ctc_wer") if key in row))\n    if not valid:\n        raise RuntimeError(f"Existing {mode}/{seed} report is not a verified completed FP32 pilot; preserve it and share the ZIP")\n    return result\n\n\ndef runtime():\n    import torch\n    import nemo\n    import pytorch_lightning as pl\n    if sys.version_info[:2] != (3, 11) or torch.__version__.split("+")[0] != "2.2.2":\n        raise RuntimeError("Use the isolated Python 3.11 / PyTorch 2.2.2 environment in the notebook")\n    source = Path(os.environ["ITANTRA_NEMO_SOURCE"]).resolve()\n    revision = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()\n    if revision != NEMO_REVISION or not Path(nemo.__file__).resolve().is_relative_to(source):\n        raise RuntimeError("Imported NeMo is not the pinned AI4Bharat source checkout")\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is unavailable; this notebook does not train on CPU")\n    gpu = torch.cuda.get_device_properties(0)\n    if gpu.total_memory < 14 * 1024**3:\n        raise RuntimeError("Use a CUDA GPU with at least 14 GiB reported VRAM for this pilot")\n    return {"python": sys.version.split()[0], "torch": torch.__version__, "lightning": pl.__version__,\n            "nemo_revision": revision, "gpu": gpu.name, "gpu_total_bytes": gpu.total_memory,\n            "cuda": torch.version.cuda, "gpu_count": torch.cuda.device_count(), "devices_used": 1,\n            "precision": "bf16-mixed" if torch.cuda.is_bf16_supported() else "16-mixed"}\n\n\ndef rnnt_cuda_preflight():\n    """Exercise the exact CUDA interop/loss path before loading a large model."""\n    import importlib.metadata as metadata\n    import torch\n    from numba import cuda\n    from numba.cuda.cudadrv import driver\n    from nemo.collections.asr.losses.rnnt import RNNTLoss\n    if not driver.USE_NV_BINDING or metadata.version("cuda-python") != "12.4.0":\n        raise RuntimeError("Install cuda-python==12.4.0 and set NUMBA_CUDA_USE_NVIDIA_BINDING=1 before importing Numba")\n    checked = runtime()\n    checked.update(numba=metadata.version("numba"), cuda_python=metadata.version("cuda-python"),\n                   numba_cuda_use_nvidia_binding=bool(driver.USE_NV_BINDING))\n    torch.manual_seed(17)\n    values = torch.randn(1, 4, 3, 5, device="cuda:0")\n    targets = torch.tensor([[1, 2]], device="cuda:0", dtype=torch.long)\n    input_lengths = torch.tensor([4], device="cuda:0", dtype=torch.long)\n    target_lengths = torch.tensor([2], device="cuda:0", dtype=torch.long)\n    results = []\n    for dtype in (torch.float32, torch.float16):\n        logits = values.to(dtype).detach().requires_grad_(True)\n        cuda.as_cuda_array(logits.detach())\n        criterion = RNNTLoss(num_classes=4, reduction="mean_batch", loss_name="warprnnt_numba")\n        loss = criterion(log_probs=logits, targets=targets, input_lengths=input_lengths,\n                         target_lengths=target_lengths)\n        loss.backward()\n        torch.cuda.synchronize()\n        if (not torch.isfinite(loss).all() or logits.grad is None\n                or not torch.isfinite(logits.grad).all() or not torch.count_nonzero(logits.grad)):\n            raise RuntimeError("RNNT CUDA smoke test produced invalid loss or gradients")\n        results.append({"dtype": str(dtype), "loss": float(loss.detach()),\n                        "gradient_norm": float(logits.grad.float().norm())})\n    return dict(checked, status="Tiny RNNT CUDA forward/backward passed; pilot not run", checks=results)\n\n\ndef extract_text(value):\n    if isinstance(value, tuple):\n        value = value[0]\n    while isinstance(value, list) and len(value) == 1:\n        value = value[0]\n    if hasattr(value, "text"):\n        value = value.text\n    if not isinstance(value, str):\n        raise RuntimeError(f"Unexpected NeMo transcript structure: {type(value).__name__}")\n    return value\n\n\ndef baseline_gate(summary):\n    return (summary["count"] == 30 and summary["native_script_count"] == 30\n            and math.isfinite(summary["wer"])\n            and abs(summary["wer"] - REFERENCE_WER) <= BASELINE_TOLERANCE)\n\n\ndef restore(checkpoint, device, trainer=None):\n    from omegaconf import open_dict\n    from nemo.collections.asr.models import ASRModel\n    config = ASRModel.restore_from(str(checkpoint), return_config=True)\n    # Saved source training paths are not valid in a new Kaggle session.\n    with open_dict(config):\n        config.train_ds = None\n        # The pinned BPE transcribe loader still reads validation_ds.get().\n        config.validation_ds = {\n            "use_start_end_token": (config.validation_ds or {}).get("use_start_end_token", False),\n            "manifest_filepath": None, "defer_setup": True,\n        }\n        config.test_ds = None\n    model = ASRModel.restore_from(str(checkpoint), override_config_path=config,\n                                 map_location=device, trainer=trainer, strict=True)\n    if model.cfg.preprocessor.sample_rate != 16000:\n        raise RuntimeError("The restored model is not configured for 16-kHz speech")\n    return model\n\n\ndef evaluation_wav_duration(path):\n    """Read protected PCM16/float32 evaluation WAVs without converting them."""\n    import numpy as np\n    import soundfile as sf\n    with sf.SoundFile(str(path)) as audio:\n        if (audio.samplerate != 16000 or audio.channels != 1 or audio.format != "WAV"\n                or audio.subtype not in ("PCM_16", "FLOAT") or audio.frames <= 0):\n            raise RuntimeError(f"Expected nonempty 16-kHz mono PCM16/float32 evaluation WAV: {path}")\n        frames, rate = audio.frames, audio.samplerate\n        samples = audio.read(dtype="float32")\n        if len(samples) != frames or not np.isfinite(samples).all():\n            raise RuntimeError(f"Truncated or non-finite evaluation WAV: {path}")\n        return frames / rate\n\n\ndef score(model, records, decoder, repo):\n    import torch\n    sys.path.insert(0, str(Path(repo) / "tools"))\n    from validate_stt import normalize, distance, peak_memory_bytes\n    from prepare_speech_finetune import native_text\n    model.change_decoding_strategy(decoder_type=decoder, lang_id="ta")\n    model.eval()\n    torch.cuda.reset_peak_memory_stats()\n    rows = []\n    for row in records:\n        if digest(row["file"]) != row["sha256"]:\n            raise RuntimeError("Evaluation WAV checksum mismatch")\n        duration = evaluation_wav_duration(row["file"])\n        torch.cuda.synchronize()\n        start = time.perf_counter()\n        with torch.inference_mode():\n            text = extract_text(model.transcribe([row["file"]], batch_size=1, logprobs=False, language_id="ta"))\n        torch.cuda.synchronize()\n        elapsed = (time.perf_counter() - start) * 1000\n        ref, hyp = normalize(row["reference"]), normalize(text)\n        rows.append({"row": row.get("row"), "audio_sha256": row["sha256"], "reference": row["reference"],\n                     "output": text, "native_script": native_text("ta", text), "transcribe_ms": elapsed,\n                     "audio_seconds": duration, "word_errors": distance(ref.split(), hyp.split()),\n                     "words": len(ref.split()), "char_errors": distance(ref, hyp), "chars": len(ref)})\n    times = sorted(r["transcribe_ms"] for r in rows)\n    return {"decoder": decoder, "summary": {"count": len(rows),\n            "wer": sum(r["word_errors"] for r in rows) / sum(r["words"] for r in rows),\n            "cer": sum(r["char_errors"] for r in rows) / sum(r["chars"] for r in rows),\n            "native_script_count": sum(r["native_script"] for r in rows),\n            "gpu_transcribe_ms_mean_including_loading": statistics.mean(times),\n            "gpu_transcribe_ms_p95_including_loading": times[math.ceil(.95 * len(times)) - 1],\n            "rtf_including_loading": sum(times) / 1000 / sum(r["audio_seconds"] for r in rows),\n            "peak_gpu_allocated_bytes": torch.cuda.max_memory_allocated(),\n            "peak_process_ram_bytes": peak_memory_bytes()}, "rows": rows}\n\n\ndef freeze_layers(model, mode):\n    layers = model.encoder.layers\n    if len(layers) < 2 or mode not in ("freeze-lower", "full"):\n        raise RuntimeError("Unexpected encoder layers or freeze mode")\n    for param in model.parameters():\n        param.requires_grad_(True)\n    frozen = len(layers) // 2 if mode == "freeze-lower" else 0\n    for layer in list(layers)[:frozen]:\n        for param in layer.parameters():\n            param.requires_grad_(False)\n    return {"encoder_layers": len(layers), "frozen_encoder_layers": frozen,\n            "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),\n            "total_parameters": sum(p.numel() for p in model.parameters())}\n\n\ndef run_baseline(repo, report):\n    import torch\n    from huggingface_hub import hf_hub_download\n    if not os.environ.get("HF_TOKEN"):\n        raise RuntimeError("Enable HF_TOKEN in Kaggle Secrets and accept the Tamil model\'s access terms")\n    checkpoint = Path(hf_hub_download(MODEL_REPO, MODEL_FILE, revision=MODEL_REVISION,\n                                     cache_dir=str(Path(repo) / "model-cache")))\n    report.update(checkpoint=str(checkpoint), checkpoint_bytes=checkpoint.stat().st_size,\n                  checkpoint_sha256=digest(checkpoint), model_repo=MODEL_REPO, model_revision=MODEL_REVISION)\n    model = restore(checkpoint, torch.device("cuda:0"))\n    report["model_class"] = type(model).__name__\n    manifest = Path(repo) / "tools/stt_models/five-language-validation/manifest-ta-30.json"\n    records = json.loads(manifest.read_text(encoding="utf-8"))\n    report["ctc"] = score(model, records, "ctc", repo)\n    write_json(Path(repo) / "baseline-report.json", report)\n    report["rnnt"] = score(model, records, "rnnt", repo)\n    report["ctc_baseline_gate_passed"] = baseline_gate(report["ctc"]["summary"])\n    report["reference_wer"] = REFERENCE_WER\n    report["maximum_absolute_gap"] = BASELINE_TOLERANCE\n    if not report["ctc_baseline_gate_passed"]:\n        raise RuntimeError("Tamil native-script/CTC baseline differs from the recorded desktop baseline; review before training")\n    report["status"] = "Tamil source CTC/RNNT baseline measured; pilot not run"\n\n\ndef run_pilot(repo, plan, report, mode, seed):\n    import torch\n    import pytorch_lightning as pl\n    from pytorch_lightning.callbacks import EarlyStopping\n    from pytorch_lightning.loggers import CSVLogger\n    from omegaconf import OmegaConf\n    from nemo.collections.asr.modules import SpectrogramAugmentation\n    baseline = json.loads((Path(repo) / "baseline-report.json").read_text(encoding="utf-8"))\n    if (not baseline.get("ctc_baseline_gate_passed") or not baseline_gate(baseline["ctc"]["summary"])\n            or baseline.get("plan_sha256") != digest(Path(repo) / "pilot-plan.json")\n            or baseline["model_revision"] != MODEL_REVISION\n            or digest(baseline["checkpoint"]) != baseline["checkpoint_sha256"]):\n        raise RuntimeError("A passing baseline from these exact prepared files/checkpoint is required")\n    if seed not in SEEDS or mode not in plan["modes"]:\n        raise RuntimeError("Only the bounded two-seed freeze/full pilot is supported")\n    run_dir = Path(repo) / f"pilot-{mode}-{seed}"\n    if (run_dir / "report.json").exists():\n        raise RuntimeError("Run already has a report; preserve it instead of overwriting")\n    run_dir.mkdir(exist_ok=True)\n    # Apply the seed to Python, NumPy, Torch and data workers, not just a folder name.\n    pl.seed_everything(seed, workers=True)\n    report.update(seed=seed, mode=mode, max_steps=MAX_STEPS, plan_sha256=digest(Path(repo) / "pilot-plan.json"),\n                  baseline_report_sha256=digest(Path(repo) / "baseline-report.json"))\n    history = []\n\n    class Evidence(pl.Callback):\n        def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\n            loss = float(outputs["loss"].detach().float().cpu())\n            if not math.isfinite(loss):\n                raise RuntimeError("Non-finite training loss")\n            history.append({"step": trainer.global_step, "loss": loss})\n            report.update(status="Pilot running; final checks not passed", history=history,\n                          global_step=trainer.global_step)\n            write_json(run_dir / "report.json", report)\n\n        def on_before_optimizer_step(self, trainer, module, optimizer):\n            if any(not torch.isfinite(p.grad).all() for p in module.parameters() if p.grad is not None):\n                raise RuntimeError("Non-finite gradients; review precision/runtime")\n\n        def on_validation_end(self, trainer, module):\n            if trainer.sanity_checking:\n                return\n            metric = trainer.callback_metrics.get("val_wer_ctc")\n            if metric is None or not math.isfinite(float(metric)):\n                raise RuntimeError("Missing/non-finite val_wer_ctc; cannot claim dev evaluation")\n            history.append({"step": trainer.global_step, "dev_ctc_wer": float(metric)})\n            write_json(run_dir / "report.json", report)\n\n    # FP16 produced non-finite gradients before the first update in the reviewed pilot.\n    report["training_precision"] = "32-true"\n    trainer = pl.Trainer(accelerator="gpu", devices=1, precision=report["training_precision"],\n                         max_steps=MAX_STEPS, max_epochs=10, accumulate_grad_batches=4,\n                         gradient_clip_val=1.0, val_check_interval=40, check_val_every_n_epoch=None,\n                         log_every_n_steps=1, num_sanity_val_steps=0, enable_checkpointing=False,\n                         logger=CSVLogger(str(run_dir), name="metrics"),\n                         callbacks=[Evidence(), EarlyStopping(monitor="val_wer_ctc", mode="min", patience=3,\n                                                             check_finite=True, strict=True)])\n    model = restore(baseline["checkpoint"], torch.device("cpu"), trainer=trainer)\n    report["parameters"] = freeze_layers(model, mode)\n    if not 0 < model.ctc_loss_weight < 1:\n        raise RuntimeError("Source model must preserve both RNNT and CTC training losses")\n    model.change_decoding_strategy(decoder_type="ctc", lang_id="ta")\n    multi = "multisoftmax" in model.cfg.decoder\n    for side in ("train", "dev"):\n        cfg = {"manifest_filepath": plan["manifests"][side], "sample_rate": 16000, "batch_size": 1,\n               "shuffle": side == "train", "num_workers": 0, "pin_memory": True,\n               "min_duration": 1, "max_duration": 10, "trim_silence": False, "is_tarred": False,\n               "return_sample_id": multi, "return_language_id": multi}\n        if side == "train":\n            cfg["augmentor"] = {"speed": {"prob": .5, "sr": 16000, "resample_type": "kaiser_fast",\n                                          "min_speed_rate": .9, "max_speed_rate": 1.1, "num_rates": 3,\n                                          "rng": seed}}\n            model.setup_training_data(OmegaConf.create(cfg))\n        else:\n            model.setup_validation_data(OmegaConf.create(cfg))\n    batch = next(iter(model.train_dataloader()))\n    if multi and (len(batch) != 6 or any(lang != "ta" for lang in batch[-1])):\n        raise RuntimeError("Actual dataloader did not return the six-field Tamil batch required by the fork")\n    model.spec_augmentation = SpectrogramAugmentation(freq_masks=2, time_masks=2, freq_width=16, time_width=.05)\n    model.setup_optimization(OmegaConf.create({"name": "adamw", "lr": 2e-5, "betas": [.9, .98],\n                                              "weight_decay": .001,\n                                              "sched": {"name": "CosineAnnealing", "warmup_steps": 2,\n                                                        "min_lr": 1e-6, "max_steps": MAX_STEPS}}))\n    def fingerprint():\n        return hashlib.sha256(b"".join(p.detach().float().flatten()[:64].cpu().numpy().tobytes()\n                              for p in model.parameters() if p.requires_grad)).hexdigest()\n    before = fingerprint()\n    OmegaConf.save(model.cfg, run_dir / "effective-model-config.yaml")\n    report["ctc_loss_weight"] = float(model.ctc_loss_weight)\n    report["actual_batch_fields"] = len(batch)\n    del batch\n    torch.cuda.reset_peak_memory_stats()\n    start = time.perf_counter()\n    try:\n        trainer.fit(model)\n        trainer.validate(model, verbose=False)\n    finally:\n        report["history"] = history\n        report["elapsed_seconds"] = time.perf_counter() - start\n        report["global_step"] = trainer.global_step\n        report["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated()\n        report["peak_gpu_reserved_bytes"] = torch.cuda.max_memory_reserved()\n    report["weights_changed"] = before != fingerprint()\n    if not report["weights_changed"] or trainer.global_step <= 0 or not any("dev_ctc_wer" in r for r in history):\n        raise RuntimeError("Pilot did not produce actual weight updates and measured dev CTC WER")\n    # Pilot weights are deliberately not exported or promoted. Keep a resumable\n    # checkpoint in private output; this is a final pilot snapshot, not a best model.\n    trainer.save_checkpoint(str(run_dir / "last-pilot.ckpt"))\n    sys.path.insert(0, str(Path(repo) / "tools"))\n    from validate_stt import peak_memory_bytes\n    report["peak_process_ram_bytes"] = peak_memory_bytes()\n    report["status"] = "Bounded pilot executed; full trial and held-out comparison not run"\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("stage", choices=("baseline", "pilot"))\n    parser.add_argument("--repo", type=Path, required=True)\n    parser.add_argument("--mode", choices=("freeze-lower", "full"))\n    parser.add_argument("--seed", type=int, choices=SEEDS)\n    args = parser.parse_args()\n    if args.stage == "pilot" and (args.mode is None or args.seed is None):\n        parser.error("Pilot requires --mode and --seed")\n    output = (args.repo / "baseline-report.json" if args.stage == "baseline"\n              else args.repo / f"pilot-{args.mode}-{args.seed}/report.json")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    if output.exists():\n        raise RuntimeError("Preserve the existing report; use a new notebook session for a fresh run")\n    report = {"status": "Not run", "language": "ta", "stage": args.stage,\n              "full_trial": "Not run", "heldout_after_training": "Not run", "export": "Not run",\n              "phone_accuracy": "Not verified", "app_promotion": "Not done"}\n    try:\n        plan = verify_plan(args.repo)\n        report["plan_sha256"] = digest(args.repo / "pilot-plan.json")\n        report["runtime"] = runtime()\n        if args.stage == "baseline":\n            run_baseline(args.repo, report)\n        else:\n            run_pilot(args.repo, plan, report, args.mode, args.seed)\n    except Exception as exc:\n        report["status"] = "Failed; stop and review report"\n        # Avoid logging arbitrary library errors that might include a credential.\n        message = str(exc).replace(os.environ.get("HF_TOKEN") or "<no-token>", "[REDACTED]")\n        report["error"] = {"type": type(exc).__name__, "message": message[:2000]}\n        report["error"]["traceback"] = traceback.format_exc().replace(\n            os.environ.get("HF_TOKEN") or "<no-token>", "[REDACTED]")[-12000:]\n        raise RuntimeError(report["error"]["message"]) from None\n    finally:\n        write_json(output, report)\n        print("Report saved:", output, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')

sys.path.insert(0, str(work))
import kathbath_gate_b_data as data_guard
import kathbath_tamil_pilot as pilot
pack_reports()


## 1. Recheck saved data and hardware
This reads and hashes saved WAVs, but downloads or copies no audio. Missing, modified, overlapping or incorrectly sampled audio stops the run. Only the reviewed Tamil output is accepted. The original protection indexes remain active.


In [ ]:
def preflight():
    global data_report
    saved = data_guard.find_saved_root('/kaggle/input')
    data_report = data_guard.prepare(saved, work)
    (work/'data-report.json').write_text(json.dumps(data_report, indent=2), encoding='utf-8')
    plan = pilot.make_plan(work, data_report)
    summary['pilot_counts'] = plan['counts']
    print('Reused saved data:', saved)
    print('Pilot subset hours:', plan['hours'], 'speakers:', plan['speakers'])
    command(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], 'gpu-preflight')
    free = shutil.disk_usage(work).free
    summary['filesystem_free_bytes_before_setup'] = free
    if free < 15 * 1024**3:
        raise RuntimeError('Need at least 15 GiB free for the isolated environment and private pilot checkpoints')
    from kaggle_secrets import UserSecretsClient
    if not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    assert os.environ.get('HF_TOKEN'), 'Enable the HF_TOKEN secret for this notebook'
    summary['hf_token_present'] = True
    print('Data recheck passed; token present, value never displayed. GPU compatibility not yet verified.')
stage('saved-data-preflight', preflight)


## 2. Isolated Python 3.11 / CUDA runtime
The kernel stays unchanged. The training subprocess uses pinned PyTorch 2.2.2 CUDA 12.1 wheels and the pinned AI4Bharat NeMo fork. These are **candidate compatibility settings** until installation, imports and the pilot actually pass. Package-resolution, CUDA/RNNT kernel or checkpoint access errors are saved; they are not bypassed.


In [ ]:
(work/'constraints.txt').write_text('torch==2.2.2\ntorchaudio==2.2.2\ntorchvision==0.17.2\nnumpy==1.26.4\nnumba==0.59.1\ncuda-python==12.4.0\nllvmlite==0.42.0\nscipy==1.13.1\nlibrosa==0.10.2.post1\npytorch-lightning==2.2.5\ntorchmetrics==1.3.2\ntransformers==4.40.2\ntokenizers==0.19.1\nhuggingface-hub==0.23.2\nhydra-core==1.3.2\nomegaconf==2.3.0\nlhotse==1.25.0\npyannote.core==5.0.0\npyannote.metrics==3.2.1\ndatasets==2.19.2\npandas==2.2.2\nsentencepiece==0.1.99\nsetuptools==69.5.1\n', encoding='utf-8')
def setup_runtime():
    global train_python
    os.environ['NUMBA_CUDA_USE_NVIDIA_BINDING'] = '1'
    os.environ['PIP_NO_CACHE_DIR'] = '1'
    os.environ['UV_NO_CACHE'] = '1'
    os.environ['UV_PYTHON_INSTALL_DIR'] = str(work/'managed-python')
    os.environ['PYTHONNOUSERSITE'] = '1'
    command([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'uv==0.8.22'], 'uv-install')
    command([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], 'python-install')
    venv = work/'venv'
    command([sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.11', str(venv)], 'venv-create')
    train_python = venv/'bin/python'
    command([train_python, '-m', 'pip', 'install', 'pip==24.0', 'setuptools==69.5.1', 'wheel==0.43.0'], 'build-tools')
    command([train_python, '-m', 'pip', 'install', 'torch==2.2.2', 'torchaudio==2.2.2', 'torchvision==0.17.2',
             '--index-url', 'https://download.pytorch.org/whl/cu121'], 'torch-install')
    source = work/'NeMo'
    if not (source/'.git').exists():
        command(['git', 'clone', '--filter=blob:none', '--no-checkout',
                 'https://github.com/AI4Bharat/NeMo.git', str(source)], 'nemo-clone')
    command(['git', '-C', source, 'checkout', '--detach', pilot.NEMO_REVISION], 'nemo-checkout')
    os.environ['ITANTRA_NEMO_SOURCE'] = str(source)
    command([train_python, '-m', 'pip', 'install', '--no-build-isolation', '-c', work/'constraints.txt', '-e', str(source)+'[asr]'], 'nemo-install')
    command([train_python, '-m', 'pip', 'install', '-c', work/'constraints.txt', 'cuda-python==12.4.0'], 'cuda-binding-install')
    command([train_python, '-m', 'pip', 'check'], 'pip-check')
    command([train_python, '-c', "import json; import kathbath_tamil_pilot as p; "
             "p.write_json('runtime-report.json', p.runtime()); print('Pinned imports and CUDA checks passed')"], 'runtime-check')
    command([train_python, '-c', "import kathbath_tamil_pilot as p; p.verify_plan('.'); "
             "p.write_json('rnnt-cuda-check-report.json', p.rnnt_cuda_preflight()); print('Tiny RNNT CUDA forward/backward passed')"], 'rnnt-cuda-check')
    command([train_python, '-m', 'pip', 'freeze'], 'installed-versions')
    summary['runtime'] = json.loads((work/'runtime-report.json').read_text())
# The subprocess must find the embedded helper modules and write reports here.
os.environ['PYTHONPATH'] = str(work)
os.chdir(work)
stage('runtime-setup', setup_runtime)


## 3. Measure original CTC and RNNT baseline
Both heads are measured on the protected 30-clip FLEURS set. The CTC gate requires 30 native-script outputs and a WER within **3 percentage points** of the recorded 21.8274% desktop baseline. This tolerance permits an FP32/PyTorch-vs-INT8 runtime comparison; it is not a training success criterion. Timings here are GPU transcription wall time including audio loading, not Android latency. The independent 100-clip and 1,377-clip noisy sets remain unused in the pilot.


In [ ]:
stage('source-baseline', lambda: command([train_python, work/'kathbath_tamil_pilot.py',
      'baseline', '--repo', work], 'source-baseline'))
baseline = json.loads((work/'baseline-report.json').read_text(encoding='utf-8'))
print('Actual CTC baseline:', baseline['ctc']['summary'])
print('Actual RNNT baseline:', baseline['rnnt']['summary'])
assert baseline['ctc_baseline_gate_passed'], 'Stop: baseline needs review before pilot'


## 4. Bounded freeze/full pilot, two seeds
Four separate processes restore the same original checkpoint. Each applies its seed, trains both RNNT/CTC heads, checks finite loss/gradients, and measures development CTC WER. Half the lower encoder blocks are frozen only in the freeze-lower mode. Batch size is 1, accumulation is 4; speed perturbation and SpecAugment are training-only. A missing Tamil batch ID or CTC dev metric fails explicitly. Twenty steps tests training readiness; it is too little to conclude accuracy improvement. Private last-pilot checkpoints remain outside the report ZIP and are not production models.


In [ ]:
def run_pilots():
    for mode in ('freeze-lower', 'full'):
        for seed in pilot.SEEDS:
            if pilot.completed_pilot(work, mode, seed) is not None:
                print('Keeping completed FP32 pilot:', mode, seed)
                continue
            if shutil.disk_usage(work).free < 3 * 1024**3:
                raise RuntimeError('Too little free disk for the next private pilot checkpoint; stop and preserve reports')
            # Kaggle also imposes an output quota separate from filesystem free space.
            working_bytes = sum(p.stat().st_size for p in work.rglob('*') if p.is_file())
            if working_bytes > 16 * 1024**3:
                raise RuntimeError('Pilot workspace exceeds 16 GiB; preserve reports instead of risking the output quota')
            command([train_python, work/'kathbath_tamil_pilot.py', 'pilot', '--repo', work,
                     '--mode', mode, '--seed', seed], f'pilot-{mode}-{seed}')
            pack_reports()
    summary['status'] = 'Bounded two-seed pilot completed; full training and improved held-out WER not established'
stage('bounded-pilot', run_pilots)


## 5. STOP and download the reports
Download **`itantra_tamil_pilot_reports.zip`** from the notebook's Output tab. Preserve outputs before ending the session. Send the ZIP for review. On any failure, the ZIP produced by the failed stage is the correct handoff; do not rerun Gate A.

Next decisions require measured runtime, baseline agreement, finite losses, actual weight updates, dev metrics, memory and seed variation. Only after review should the bounded 10-hour trial and best-dev checkpoint selection run. The final locked comparison must cover 30-clip FLEURS, independent 100-clip FLEURS, full noisy test, FP32/INT8 export, size/speed/RAM and recorded phone tests. No <10% claim or app pack promotion is supported by this pilot.


In [ ]:
runs = []
for mode in ('freeze-lower', 'full'):
    for seed in pilot.SEEDS:
        path = work/f'pilot-{mode}-{seed}/report.json'
        result = json.loads(path.read_text(encoding='utf-8'))
        dev = [row['dev_ctc_wer'] for row in result['history'] if 'dev_ctc_wer' in row]
        assert result['weights_changed'] and dev, 'No measured pilot outcome'
        runs.append({'mode': mode, 'seed': seed, 'optimizer_steps': result['global_step'],
                     'last_subset_dev_ctc_wer': dev[-1], 'peak_gpu_allocated_bytes': result['peak_gpu_allocated_bytes'],
                     'elapsed_seconds': result['elapsed_seconds']})
comparison = {'status': 'Measured short pilot only; general accuracy improvement not established',
              'runs': runs, 'full_trial': 'Not run', 'post_training_heldout_wer': 'Not run',
              'phone_accuracy': 'Not verified', 'app_promotion': 'Not done'}
pilot.write_json(work/'pilot-comparison-report.json', comparison)
pack_reports()
print(json.dumps(comparison, indent=2))
print('STOP. Return the report ZIP before longer training. Keep the current app packs.')
